        # 05 統合比較（Stage 3：Integrated Evaluation）

        マッチングアルゴリズムを、**成立時点**（成立率・初期満足度・Blocking pair）と**長期継続**（5・10・20 年継続率、長期成立率）の両方から比べます。

        > **研究課題（RQ5）**：「成立率が最も高いアルゴリズムは、10 年後の長期成立率も最も高いか」
        > この Notebook は答えを決めていません。**実際のシミュレーション結果**から検証してください。

> **教育用の架空データ・架空モデル**です。参加者は実在の人物ではなく、Group A・B はアルゴリズム上の区分で性別や性的指向を意味しません。希望スコアは実際の結婚適性を表さず、関係継続モデルの係数は実際の離婚率や実証された因果効果ではありません。

## 準備：実験環境（最初に必ず実行）

F3 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F3 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F3 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F3 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f3-matching-algorithms"     # リポジトリの中の F3 フォルダ（ルートが F3 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f3_root(p):
    return (Path(p) / "matching_algorithms.py").exists() and (Path(p) / "relationship_stability.py").exists()


def find_f3_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F3 のルート
        if is_f3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f3_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f3_upload")
    for hit in glob.glob("/content/f3_upload/**/matching_algorithms.py", recursive=True):
        if is_f3_root(Path(hit).parent):
            return Path(hit).parent
    return None


F3_ROOT = find_f3_root()
if F3_ROOT is None:
    raise RuntimeError("F3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F3_ROOT)
if str(F3_ROOT) not in sys.path:
    sys.path.insert(0, str(F3_ROOT))
if not (F3_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。preferences.py・matching_algorithms.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import participants as pt
import preferences as pr
import matching_algorithms as ma
import relationship_stability as rs
import evaluation as ev
import matching_charts as mc

mc.setup_style()                       # 文字を大きくするなら mc.setup_style(1.3)
groups = pt.load_participants()
scores = pr.score_table(groups)
rankings = pr.build_rankings(groups, scores)
params = rs.load_parameters()          # 関係継続モデルのパラメータ（すべて架空）
A_IDS = [p["id"] for p in groups["A"]]
B_IDS = [p["id"] for p in groups["B"]]
N_TOTAL = pt.total_participants(groups)
compat = lambda a, b: pr.pair_compatibility(a, b, scores)

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", mc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F3_ROOT:", F3_ROOT)
print(f"参加者 Group A {len(A_IDS)} 人・Group B {len(B_IDS)} 人 / 関係継続モデル（架空）:",
      {k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                               "default_conflict_resolution", "simulation_years", "random_seed")})

## 比較の条件（公平性）

- 同じ参加者・同じ希望順位・同じ受け入れ可能性
- 同じ関係継続モデルの係数・観測期間・反復回数
- Random はマッチング自体が乱数で変わるので、`MATCHING_SEEDS` の平均
- **共通乱数**：同じペア・同じ反復なら、どのアルゴリズムで成立しても同じ乱数（違うペアの乱数は独立）

$$ \text{Long-term Matching Rate}(T) = \frac{2 \times \text{T 年後も継続しているペア数}}{\text{全参加者数}} = \text{成立率} \times \text{継続率}(T) $$

In [ ]:
MATCHING_SEEDS = range(20)              # TODO：変えてみよう
REPLICATES = 300                        # TODO：変えてみよう
res = ev.compare_algorithms(groups, rankings, scores, params, MATCHING_SEEDS, REPLICATES)
table(list(res.values()), [("label", "algorithm", "{}"), ("matching_rate", "match rate", "{:.0%}"),
                           ("satisfaction", "satisfaction", "{:.2f}"), ("blocking_pairs", "blocking", "{:.2f}"),
                           ("mean_compatibility", "compat.", "{:.3f}"), ("continuation_5y", "5y cont.", "{:.1%}"),
                           ("continuation_10y", "10y cont.", "{:.1%}"), ("continuation_20y", "20y cont.", "{:.1%}"),
                           ("long_term_10y", "10y long-term", "{:.1%}")])

## 1〜2. 成立時点：成立率と初期満足度（誤差棒は Random のシード間の標準偏差）

In [ ]:
#@title グラフを表示（コードは隠しています）
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
mc.plot_metric_bars(res, "matching_rate", "Matching rate", "rate", err_key="matching_rate_seed_sd", ax=axes[0], ylim=(0, 1.1))
mc.plot_metric_bars(res, "satisfaction", "Initial satisfaction", "satisfaction", fmt="{:.2f}",
                    err_key="satisfaction_seed_sd", ax=axes[1], ylim=(0, 1.1))
mc.plot_metric_bars(res, "blocking_pairs", "Blocking pairs", "count", fmt="{:.1f}", err_key="blocking_pairs_seed_sd", ax=axes[2])
fig.tight_layout()
show(fig, "05_formation")

## 成立ペアの初期適合度の分布（RQ3）

In [ ]:
#@title グラフを表示（コードは隠しています）
dist = {name: [compat(a, b) for run in r["runs"] for a, b in run["matching"].items()] for name, r in res.items()}
show(mc.plot_compat_distribution(dist, "Initial compatibility of matched pairs (all runs)"), "05_compatibility")

## 3. アルゴリズム別の継続率曲線（架空モデル）

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_survival({r["label"]: r["survival_curve"] for r in res.values()},
                      f"Continuation of matched pairs by algorithm (model, {REPLICATES} replicates)",
                      colors={r["label"]: mc.ALG_COLORS[n] for n, r in res.items()}), "05_survival")

## 4〜5. 10 年長期成立率と、成立率との比較

In [ ]:
show(mc.plot_grouped(res, [("matching_rate", "matching rate (at matching)"), ("long_term_5y", "long-term 5y"),
                           ("long_term_10y", "long-term 10y"), ("long_term_20y", "long-term 20y")],
                     "Matching rate vs long-term matching rate (model)"), "05_long_term")
rank_now = [r["label"] for r in sorted(res.values(), key=lambda r: -r["matching_rate"])]
rank_10y = [r["label"] for r in sorted(res.values(), key=lambda r: -r["long_term_10y"])]
print("成立率の順位        :", " > ".join(rank_now))
print("10 年長期成立率の順位:", " > ".join(rank_10y))
print("※ 同じ値（同率）の場合は並び順に意味はない。差が反復・シードのばらつきより大きいかを確かめること")

## 6. モデル係数の感度分析

初期適合度の影響係数 `alpha` を変えると、アルゴリズム間の 10 年継続率の差はどう変わるか。`alpha = 0` は**対照条件**（初期適合度が解消確率に影響しないモデル、Experiment F）です。

In [ ]:
#@title グラフを表示（コードは隠しています）
alphas = [0.0, 0.02, 0.04, 0.06, 0.08]   # TODO：beta・gamma・baseline_hazard・ストレスでも試そう
series = {n: [] for n in res}
for a_ in alphas:
    r_ = ev.compare_algorithms(groups, rankings, scores, rs.with_changes(params, alpha=a_), range(10), 200)
    for n in series:
        series[n].append(r_[n]["continuation_10y"])
show(mc.plot_sensitivity(alphas, series, "alpha (effect of initial compatibility, fictional)",
                         "Sensitivity of 10-year continuation to alpha (alpha = 0: control)"), "05_sensitivity_alpha")

## 公平性

長期継続率だけで公平性を判断せず、グループ別の満足度・最低満足度・未マッチング率も見ます。

In [ ]:
rows = []
for name in ma.ALGORITHMS:
    m = ma.run(name, rankings, A_IDS, B_IDS, seed=0)
    s = ev.satisfaction_summary(m, rankings, A_IDS, B_IDS)
    rows.append({"alg": ma.LABELS[name] + (" (seed 0)" if name == "random" else ""), **s,
                 "ranks": dict(sorted(s["rank_distribution"].items(), key=lambda kv: str(kv[0])))})
table(rows, [("alg", "algorithm", "{}"), ("mean_A", "mean A", "{:.2f}"), ("mean_B", "mean B", "{:.2f}"),
             ("min_matched", "min (matched)", "{:.2f}"), ("unmatched_A", "unmatched A", "{:.0%}"),
             ("unmatched_B", "unmatched B", "{:.0%}"), ("ranks", "partner rank: count", "{}")])

## 学生向け TODO・考察

- [ ] `MATCHING_SEEDS`・`REPLICATES` を変えて、順位が安定するか確かめる（単一のシードで断定しない）
- [ ] 希望条件を厳しくした場合（Experiment A）にも、同じ結論になるか調べる
- [ ] 対照条件（alpha = 0）で、アルゴリズム間の継続率の差がどうなるか説明する

**考察（チームで記入）**：
- **TODO**：RQ5 への答え（成立率の順位と 10 年長期成立率の順位は同じか。差はばらつきより大きいか）
- **TODO**：RQ6 への答え（Blocking pair が 0 であることと、継続率の関係）
- **TODO**：結論が、モデルのどの仮定（alpha など）に依存しているか

## 参考文献

- D. Gale, L. S. Shapley, "College Admissions and the Stability of Marriage", *The American Mathematical Monthly*, 69(1), 9–15, 1962.
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第3章 線形探索・ハッシュ法、第4章 キュー、第6章 挿入ソート）
- F3 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_participants.json`、`data/relationship_parameters.json`（この教材のための**架空のデータ**）